# 1.0 Setup: S3, Glue, Athena (idempotent, no IAM create calls)

In [ ]:
import sys, pathlib, boto3
from botocore.exceptions import ClientError, NoCredentialsError
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pamap2.config import REGION, build_config, save_config

session = boto3.Session(region_name=REGION)
try:
    ident = session.client("sts").get_caller_identity()
except (NoCredentialsError, ClientError) as e:
    raise SystemExit(f"AWS authentication failed (check credentials / region {REGION}): {e}")
account_id = ident["Account"]
print("Authenticated as", ident["Arn"], "in", REGION)
cfg = build_config(account_id)

In [ ]:
# Role: reuse existing LabRole (never create IAM resources)
try:
    import sagemaker
    exec_role = sagemaker.get_execution_role()
    if exec_role.split("/")[-1] != "LabRole":
        print("WARNING: execution role differs from LabRole:", exec_role.split("/")[-1])
    else:
        print("Execution role matches LabRole")
except Exception as e:
    print("Could not cross-check execution role:", e)
print("Using role ARN:", cfg["role_arn"])

In [ ]:
s3 = session.client("s3")
bucket = cfg["bucket"]
try:
    s3.head_bucket(Bucket=bucket)
    print("bucket exists, reusing:", bucket)
except ClientError as e:
    if e.response["Error"]["Code"] not in ("404", "NoSuchBucket", "NotFound"):
        raise
    try:
        s3.create_bucket(Bucket=bucket)  # us-east-1: no LocationConstraint
        print("created bucket:", bucket)
    except ClientError as e2:
        if e2.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
            raise
        print("bucket exists, reusing:", bucket)
s3.put_public_access_block(Bucket=bucket, PublicAccessBlockConfiguration={
    "BlockPublicAcls": True, "IgnorePublicAcls": True,
    "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
s3.put_bucket_encryption(Bucket=bucket, ServerSideEncryptionConfiguration={
    "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})
for prefix in cfg["s3_prefixes"].values():
    try:
        s3.head_object(Bucket=bucket, Key=prefix)
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
        s3.put_object(Bucket=bucket, Key=prefix)
        print("created prefix", prefix)
print("prefixes ready")

In [ ]:
glue = session.client("glue")
try:
    glue.create_database(DatabaseInput={"Name": cfg["glue_database"], "Description": "PAMAP2 HAR data lake"})
    print("created Glue database", cfg["glue_database"])
except glue.exceptions.AlreadyExistsException:
    print("Glue database exists, reusing:", cfg["glue_database"])

In [ ]:
athena = session.client("athena")
try:
    athena.create_work_group(
        Name=cfg["athena_workgroup"],
        Description="PAMAP2 HAR queries",
        Configuration={"ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/{cfg['s3_prefixes']['athena-results']}",
            "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"}},
            "EnforceWorkGroupConfiguration": True})
    print("created workgroup", cfg["athena_workgroup"])
except ClientError as e:
    if e.response["Error"]["Code"] != "InvalidRequestException" or "already created" not in e.response["Error"]["Message"]:
        raise
    print("workgroup exists, reusing:", cfg["athena_workgroup"])

In [ ]:
path = save_config(cfg)
print("Wrote", path)